In [ ]:
import numpy as np
import ssqpy
from time import time, sleep

np.set_printoptions(suppress=True)

# Build MPC
ssqpy.setSilentMode()

dt = 0.01
MPC_H = 15

V_WGT = 3e-2
U_WGT = 2e-3
TIP_WGT = 4.2
J1_WGT = 0.0
J2_WGT = 2.0

torque_clip = 0.1

model = ssqpy.model.Model(
    MPC_H,
    dt,
    urdf_path="pendubot.urdf",
    actuated_joints=[0],
    solver_mode=ssqpy.model.SolverMode.InverseDynamics
)

nq = model.getnq()
nv = model.getnv()
nu = model.getnu()

vel_cost = ssqpy.model.costs.SquaredJointVelocityCost(model, V_WGT)
u_cost = ssqpy.model.costs.SquaredControlCost(model, U_WGT)
tip_cost = ssqpy.model.costs.FrameSquaredTranslationErrorCost(
    model, "tip", np.array((0.0, 0.0, 0.2)), TIP_WGT
)
config_cost = ssqpy.model.costs.SquaredConfigurationErrorCost(
    model, np.array((np.pi, 0.0)), np.array((J1_WGT, J2_WGT))
)

for k in range(MPC_H):
    model.addCost(k, vel_cost)
    model.addCost(k, u_cost)
    model.addCost(k, tip_cost)
    model.addCost(k, config_cost)

model.addCost(MPC_H, vel_cost)
model.addCost(MPC_H, tip_cost)
model.addCost(MPC_H, config_cost)

ssqp_params = ssqpy.solvers.ssqpParams()
ssqp_params.tolerance = 1e-1

admm_params = ssqpy.solvers.admmParams()
admm_params.abs_tolerance = 1e-2
admm_params.rel_tolerance = 1e-2
admm_params.warm_start = True
ssqp_params.admmParams = admm_params


mpc = ssqpy.solvers.MPC(
    model, ssqp_params, sqp_iters=4, qp_iters=100
)

In [ ]:
from cloudpendulumclient.client import Client

with open("../token.txt", "r") as f:
    token = f.readlines()[0].strip()

Tf = 30.0
client = Client()
session_token, livestream_url = client.start_experiment(
    user_token = token,
    experiment_type = "Pendubot",
    experiment_time = Tf,
    preparation_time = 5.0,
    record = True,
    cell_id=203
)

print("Received response from server!")
print("Session token: ", session_token)
print("Livestream url: ", livestream_url)

current_time = 0.0

np.set_printoptions(suppress=True)

start_all = time()
while (time() - start_all) < Tf:
    start = time()

    mq = client.get_position(session_token)
    mv = client.get_velocity(session_token)

    try:
        u = mpc.step(np.hstack((mq, mv)))[1].stage(0)
    except RuntimeError:
        u = np.zeros(nv)

    tau = model.inverseDynamics(np.array(mq), np.array(mv), u)
    tau = np.clip(tau, -torque_clip, torque_clip)

    client.set_torque(tau[:1], session_token)

    elapsed = time() - start
        
url = client.stop_experiment(session_token)

print("Final state:", mq)